# SigLIP2 Upper — Inference & Evaluation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
SigLIP2 Upper Sub-Classifier — Inference & Evaluation
======================================================
Runs on both test datasets:
  1. classified_images_13_14...
  2. Careys_labelled_data

Handles mixed-case folder names and filters only upper classes.
"""

import os, json, random, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

MODEL_DIR = Path("/content/drive/Shareddrives/Garment Type/Complete_dataset/Siglip2_models/siglip2_upper_sub")

# Both test dataset roots — UPDATE paths if needed
TEST_DATASETS = {
    "classified_13_14": "/content/drive/Shareddrives/Garment Type/classified_images_13_14_test",
    "careys": "/content/drive/Shareddrives/Garment Type/Careys_labelled_data",
}

EVAL_DIR = MODEL_DIR / "eval_results"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Model config
SIGLIP2_MODEL_ID = "google/siglip2-so400m-patch14-384"
IMG_SIZE = 384
HIDDEN_SIZE = 1152
NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)

BATCH_SIZE = 24
NUM_WORKERS = 2

# ============================================================
# FOLDER NAME → CANONICAL CLASS MAPPING
# ============================================================
# Covers all naming variations across both test datasets
UPPER_FOLDER_MAP = {
    # blazer
    "blazer": "blazer",     "Blazer": "blazer",     "BLAZER": "blazer",
    # dresses
    "dresses": "dresses",   "Dresses": "dresses",   "DRESSES": "dresses",
    # fleece
    "fleece": "fleece",     "Fleece": "fleece",     "FLEECE": "fleece",
    # jumpers
    "jumpers": "jumpers",   "Jumpers": "jumpers",   "JUMPERS": "jumpers",
    # shirt
    "shirt": "shirt",       "Shirt": "shirt",       "SHIRT": "shirt",
    # t-shirt (many variations)
    "t-shirt": "t-shirt",   "T-shirt": "t-shirt",   "T-Shirt": "t-shirt",
    "t_shirt": "t-shirt",   "T_shirt": "t-shirt",   "T_Shirt": "t-shirt",
    "tshirt": "t-shirt",    "Tshirt": "t-shirt",
}

UPPER_CLASSES = sorted(set(UPPER_FOLDER_MAP.values()))


# ============================================================
# MODEL DEFINITION
# ============================================================
class SigLIP2Classifier(nn.Module):
    def __init__(self, model_id, num_classes, dropout_rate=0.3):
        super().__init__()
        from transformers import AutoModel
        full_model = AutoModel.from_pretrained(model_id, ignore_mismatched_sizes=True)
        self.vision = full_model.vision_model
        del full_model

        self.classifier = nn.Sequential(
            nn.LayerNorm(HIDDEN_SIZE),
            nn.Dropout(dropout_rate),
            nn.Linear(HIDDEN_SIZE, num_classes),
        )

    def forward(self, pixel_values):
        outputs = self.vision(pixel_values=pixel_values)
        pooled = outputs.pooler_output
        return self.classifier(pooled)


# ============================================================
# DATA COLLECTION WITH NAME MAPPING
# ============================================================
def collect_test_samples(root):
    """Collect test samples with folder name mapping. Flat: ROOT/class_name/images."""
    root = Path(root)
    samples = []
    skipped_folders = []

    if not root.exists():
        print(f"  ⚠️ Path does not exist: {root}")
        return samples, skipped_folders

    for folder in sorted(root.iterdir()):
        if not folder.is_dir():
            continue

        canonical = UPPER_FOLDER_MAP.get(folder.name)
        if canonical is None:
            skipped_folders.append(folder.name)
            continue

        img_files = [
            f for f in folder.iterdir()
            if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
        ]
        for img_path in img_files:
            samples.append((str(img_path), canonical))

    return samples, skipped_folders


class TestDataset(Dataset):
    def __init__(self, samples, cls2idx, img_size):
        self.samples = samples
        self.cls2idx = cls2idx
        self.img_size = img_size
        self.transform = A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(NORM_MEAN, NORM_STD),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        p, c = self.samples[i]
        try:
            img = np.array(Image.open(p).convert("RGB"))
        except Exception:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        data = self.transform(image=img)
        return data["image"], self.cls2idx[c], p


# ============================================================
# LOAD MODEL + THRESHOLDS
# ============================================================
def load_model_and_thresholds():
    ckpt_path = MODEL_DIR / "best_siglip2_upper.pt"
    assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

    print(f"Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    cls2idx = ckpt["cls2idx"]
    idx2cls = {v: k for k, v in cls2idx.items()}
    classes = ckpt["classes"]

    print(f"  Classes: {classes}")
    print(f"  Best training F1: {ckpt['best_f1']:.4f}")

    model = SigLIP2Classifier(SIGLIP2_MODEL_ID, len(cls2idx), dropout_rate=0.0).to(DEVICE)
    model.vision.load_state_dict(ckpt["vision_model"])
    model.classifier.load_state_dict(ckpt["classifier"])
    model.eval()
    print(f"  ✅ Model loaded")

    thresh_path = MODEL_DIR / "upper_siglip2_thresholds.json"
    if thresh_path.exists():
        with open(thresh_path) as f:
            thresholds = json.load(f)["thresholds"]
        print(f"\n  Thresholds:")
        for c in sorted(thresholds.keys()):
            print(f"    {c:15s}: {thresholds[c]:.2f}")
    else:
        print(f"  ⚠️ No threshold file, using 0.5")
        thresholds = {c: 0.5 for c in classes}

    return model, cls2idx, idx2cls, classes, thresholds


# ============================================================
# INFERENCE
# ============================================================
@torch.no_grad()
def run_inference(model, dataloader, idx2cls):
    all_paths, all_true, all_pred_idx, all_pred_lbl, all_conf, all_probs = [], [], [], [], [], []

    for xb, yb, paths in tqdm(dataloader, desc="Inference"):
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits = model(xb)

        probs = F.softmax(logits, dim=1).cpu().numpy()
        pred_idx = logits.argmax(1).cpu().numpy()
        conf = probs[np.arange(len(pred_idx)), pred_idx]

        all_paths.extend(paths)
        all_true.extend(yb.numpy().tolist())
        all_pred_idx.extend(pred_idx.tolist())
        all_pred_lbl.extend([idx2cls[p] for p in pred_idx])
        all_conf.extend(conf.tolist())
        all_probs.append(probs)

    return {
        "paths": all_paths,
        "true_labels": np.array(all_true),
        "pred_indices": np.array(all_pred_idx),
        "pred_labels": all_pred_lbl,
        "confidences": np.array(all_conf),
        "probs": np.concatenate(all_probs, axis=0),
    }


# ============================================================
# EVALUATION
# ============================================================
def evaluate(results, cls2idx, idx2cls, classes, thresholds, dataset_name):
    y_true = results["true_labels"]
    y_pred = results["pred_indices"]
    confidences = results["confidences"]
    paths = results["paths"]
    probs = results["probs"]

    # ── RAW METRICS ──
    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] RAW METRICS (no threshold)")
    print(f"{'='*65}")

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    print(f"\n  Accuracy:      {acc:.4f}")
    print(f"  F1 (macro):    {f1_macro:.4f}")
    print(f"  F1 (weighted): {f1_weighted:.4f}")

    rep = classification_report(y_true, y_pred, target_names=classes, digits=4)
    print(f"\n{rep}")

    cm = confusion_matrix(y_true, y_pred)
    print(f"  Confusion Matrix:")
    header = "          " + "  ".join(f"{c[:7]:>7s}" for c in classes)
    print(header)
    for i, cls_name in enumerate(classes):
        row = f"  {cls_name[:8]:>8s}  " + "  ".join(f"{cm[i,j]:>7d}" for j in range(len(classes)))
        print(row)

    # ── THRESHOLD-FILTERED ──
    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] THRESHOLD-FILTERED METRICS")
    print(f"{'='*65}")

    print(f"\n  {'Class':<15s} {'Thr':>6s} {'Total':>7s} {'Accept':>7s} {'Reject':>7s} "
          f"{'Rej%':>6s} {'Acc_raw':>8s} {'Acc_filt':>9s}")
    print(f"  {'-'*75}")

    total_accepted = 0
    total_correct_accepted = 0
    total_samples = 0

    for cls_name in classes:
        cls_idx = cls2idx[cls_name]
        thr = thresholds.get(cls_name, 0.5)
        pred_mask = (y_pred == cls_idx)
        n_total = pred_mask.sum()

        if n_total == 0:
            print(f"  {cls_name:<15s} {thr:>6.2f} {0:>7d} {0:>7d} {0:>7d} {'N/A':>6s} {'N/A':>8s} {'N/A':>9s}")
            continue

        raw_correct = (y_true[pred_mask] == cls_idx).sum()
        raw_acc = raw_correct / n_total

        conf_mask = confidences[pred_mask] >= thr
        n_accepted = conf_mask.sum()
        n_rejected = n_total - n_accepted

        if n_accepted > 0:
            filt_correct = (y_true[pred_mask][conf_mask] == cls_idx).sum()
            filt_acc = filt_correct / n_accepted
            total_correct_accepted += filt_correct
        else:
            filt_acc = 0.0

        total_accepted += n_accepted
        total_samples += n_total
        rej_pct = f"{n_rejected / n_total * 100:.1f}%"
        print(f"  {cls_name:<15s} {thr:>6.2f} {n_total:>7d} {n_accepted:>7d} {n_rejected:>7d} "
              f"{rej_pct:>6s} {raw_acc:>8.3f} {filt_acc:>9.3f}")

    if total_accepted > 0:
        print(f"\n  Overall filtered accuracy: {total_correct_accepted / total_accepted:.4f}")
        print(f"  Overall rejection rate:    {(total_samples - total_accepted) / total_samples:.4f}")
        print(f"  Accepted: {total_accepted}/{total_samples}")

    # ── CONFIDENCE DISTRIBUTION ──
    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] CONFIDENCE DISTRIBUTION")
    print(f"{'='*65}")

    correct_mask = (y_true == y_pred)
    wrong_mask = ~correct_mask

    if correct_mask.sum() > 0:
        print(f"\n  Correct: mean={confidences[correct_mask].mean():.3f}  "
              f"median={np.median(confidences[correct_mask]):.3f}  "
              f"min={confidences[correct_mask].min():.3f}")
    if wrong_mask.sum() > 0:
        print(f"  Wrong:   mean={confidences[wrong_mask].mean():.3f}  "
              f"median={np.median(confidences[wrong_mask]):.3f}  "
              f"max={confidences[wrong_mask].max():.3f}")
        gap = confidences[correct_mask].mean() - confidences[wrong_mask].mean()
        print(f"  Gap:     {gap:.3f}")

    # ── MISCLASSIFIED ──
    misclassified = []
    for i in range(len(y_true)):
        if y_true[i] != y_pred[i]:
            misclassified.append({
                "path": paths[i],
                "true_label": idx2cls[y_true[i]],
                "pred_label": idx2cls[y_pred[i]],
                "confidence": round(float(confidences[i]), 4),
                "true_class_prob": round(float(probs[i, y_true[i]]), 4),
            })
    misclassified.sort(key=lambda x: x["confidence"], reverse=True)

    print(f"\n{'='*65}")
    print(f"  [{dataset_name}] MISCLASSIFIED: {len(misclassified)} / {len(y_true)}")
    print(f"{'='*65}")

    print(f"\n  Top high-confidence mistakes:")
    print(f"  {'True':<12s} {'Pred':<12s} {'Conf':>6s} {'TrueP':>6s}  File")
    print(f"  {'-'*70}")
    for m in misclassified[:20]:
        fname = Path(m["path"]).name[:30]
        print(f"  {m['true_label']:<12s} {m['pred_label']:<12s} {m['confidence']:>6.3f} "
              f"{m['true_class_prob']:>6.3f}  {fname}")

    # ── ERROR PAIRS ──
    print(f"\n  Error pairs:")
    for cls_name in classes:
        wrong_from = [m for m in misclassified if m["true_label"] == cls_name]
        if not wrong_from:
            print(f"    {cls_name}: 0 errors ✅")
        else:
            pairs = Counter(m["pred_label"] for m in wrong_from)
            desc = ", ".join(f"{k}({v})" for k, v in pairs.most_common(3))
            print(f"    {cls_name}: {len(wrong_from)} errors → {desc}")

    # ── SAVE ──
    tag = dataset_name.replace(" ", "_")

    with open(EVAL_DIR / f"misclassified_upper_{tag}.json", "w") as f:
        json.dump(misclassified, f, indent=2)

    eval_summary = {
        "model": "SigLIP2_SO400M_Upper",
        "dataset": dataset_name,
        "total_samples": int(len(y_true)),
        "accuracy": float(acc),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "n_misclassified": len(misclassified),
        "per_class_counts": {idx2cls[i]: int((y_true == i).sum()) for i in range(len(classes))},
    }

    with open(EVAL_DIR / f"eval_summary_upper_{tag}.json", "w") as f:
        json.dump(eval_summary, f, indent=2)

    np.savez(EVAL_DIR / f"predictions_upper_{tag}.npz",
             y_true=y_true, y_pred=y_pred, confidences=confidences, probs=probs, classes=classes)
    (EVAL_DIR / f"report_upper_{tag}.txt").write_text(rep)
    np.save(EVAL_DIR / f"cm_upper_{tag}.npy", cm)

    print(f"\n  Saved → {EVAL_DIR}/*_{tag}.*")
    return eval_summary


# ============================================================
# MAIN
# ============================================================
def main():
    print(f"{'='*65}")
    print(f"  SigLIP2 Upper — Eval on Multiple Test Datasets")
    print(f"{'='*65}")
    print(f"  Device: {DEVICE}")

    model, cls2idx, idx2cls, classes, thresholds = load_model_and_thresholds()

    all_summaries = {}

    for ds_name, ds_path in TEST_DATASETS.items():
        print(f"\n\n{'#'*65}")
        print(f"  DATASET: {ds_name}")
        print(f"  PATH:    {ds_path}")
        print(f"{'#'*65}")

        test_samples, skipped = collect_test_samples(ds_path)

        if skipped:
            print(f"\n  Skipped folders (not upper): {', '.join(skipped)}")

        if not test_samples:
            print(f"\n  ❌ No upper samples found")
            continue

        test_counts = Counter(c for _, c in test_samples)
        print(f"\n  Upper samples: {len(test_samples)}")
        for cls, cnt in test_counts.most_common():
            print(f"    {cls:15s}: {cnt}")

        test_ds = TestDataset(test_samples, cls2idx, IMG_SIZE)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True)

        results = run_inference(model, test_loader, idx2cls)
        summary = evaluate(results, cls2idx, idx2cls, classes, thresholds, ds_name)
        all_summaries[ds_name] = summary

    # ── CROSS-DATASET COMPARISON ──
    if len(all_summaries) > 1:
        print(f"\n\n{'='*65}")
        print(f"  COMPARISON ACROSS DATASETS")
        print(f"{'='*65}")
        print(f"\n  {'Dataset':<25s} {'Samples':>8s} {'Acc':>8s} {'F1_macro':>9s} {'Errors':>8s}")
        print(f"  {'-'*60}")
        for name, s in all_summaries.items():
            print(f"  {name:<25s} {s['total_samples']:>8d} {s['accuracy']:>8.4f} "
                  f"{s['f1_macro']:>9.4f} {s['n_misclassified']:>8d}")

    print(f"\n\nDONE. All results in: {EVAL_DIR}")


main()